In [1]:
# ========================================================================================
# 1. PREPARACIÓN DEL ENTORNO MODERNO (Sólo para pruebas locales, borrar en caso contrario)
# ========================================================================================
using Pkg
Pkg.activate("/home/antonibancells/Desktop/Codi/TFM/ProvesModernes")

  Activating project at `~/Desktop/Codi/TFM/ProvesModernes`


In [2]:
#Librerías necesarias
using ITensors
using ITensorMPS
using Plots
using LinearAlgebra

A continuación se implementan, de manera manual, diferentes tensores de bajo rango y se verifica comparándolo con el MPS, obtenido a partir del vector denso mediante codificación binaria (QTT) y SVD. El primer paso es definir una función de distancia, evaluar_mps_manual, para poder comparar. Después se construye el MPS de las dos maneras: a partir del vector denso con SVD (función nativa en ITensor) y después la construcción manual. El primer ejemplo es f(x)=x

In [28]:
"""
Función para evaluar la calidad de tu MPS hecho a mano frente al SVD clásico.
"""
function evaluar_mps_manual(N::Int, mps_clasico::MPS, mps_manual::MPS)
    # Calcular el producto interno (solapamiento)
    # Si son idénticos, |<psi_c|psi_m>| / (||psi_c|| * ||psi_m||) debe ser exactamente 1.0
    norma_c = norm(mps_clasico)
    norma_m = norm(mps_manual)
    
    # Distancia matemática entre vectores: || ψ_c - ψ_m ||
    # Se calcula eficientemente en MPS como: sqrt(norm_c^2 + norm_m^2 - 2*real(<psi_c|psi_m>))
    solapamiento = inner(mps_clasico, mps_manual)
    distancia = sqrt(max(0.0, norma_c^2 + norma_m^2 - 2 * real(solapamiento)))

    return distancia
end

evaluar_mps_manual

In [58]:
# ==========================================================
# Construcción mediante vector denso con SVD, para f(x)=x
# ==========================================================
function construir_lineal_SVD(sites,x_min,dx,N)
  psi_tensor = ITensor(sites...)
  for i in 0:(dim_total - 1)
    x = x_min + i * dx
    val = x  # f(x) = x
    estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
    psi_tensor[estado_binario...] = val
  end
  mps_svd = MPS(psi_tensor, sites; cutoff=1e-15)
  return mps_svd
end

construir_lineal_SVD (generic function with 2 methods)

In [59]:
# ==========================================================
# Construcción manual del MPS para f(x)=x (Rango χ = 2)
# ==========================================================
function construir_lineal_manual(sites,x_min, dx, N)
    b = MPS(sites)
    blinks = [Index(2, "link,l=$i") for i in 1:(N-1)]
    
    # --- Sitio 1 (Inicio) ---
    b[1] = ITensor(sites[1], blinks[1])
    for s in 1:2
        contribucion = (s == 1) ? 0.0 : dx * 2.0^0
        # El vector inicial empuja: 
        # Canal 1 (columna 1) -> El acumulador base (siempre 1.0)
        # Canal 2 (columna 2) -> La suma acumulada actual
        b[1][sites[1]=>s, blinks[1]=>1] = 1.0
        b[1][sites[1]=>s, blinks[1]=>2] = x_min+contribucion
    end

    # --- Bulk (Sitios 2 a N-1) ---
    # Cada bloque aplica la matriz de transferencia del autómata:
    # [    1          0     ]
    # [ contribucion   1     ]
    for k in 2:(N-1)
        b[k] = ITensor(blinks[k-1], sites[k], blinks[k])
        val_bit = dx * 2.0^(k-1)
        for s in 1:2
            contribucion = (s == 1) ? 0.0 : val_bit
            
            # Fila 1 del enlace izquierdo
            b[k][blinks[k-1]=>1, sites[k]=>s, blinks[k]=>1] = 1.0 #columna 1
            b[k][blinks[k-1]=>1, sites[k]=>s, blinks[k]=>2] = contribucion #columna 2
            
            # Fila 2 del enlace izquierdo
            b[k][blinks[k-1]=>2, sites[k]=>s, blinks[k]=>1] = 0.0 #columna 1
            b[k][blinks[k-1]=>2, sites[k]=>s, blinks[k]=>2] = 1.0 #columna 2
        end
    end

    # --- Sitio N (Cierre) ---
    b[N] = ITensor(blinks[N-1], sites[N])
    val_bit = dx * 2.0^(N-1)
    for s in 1:2
        contribucion = (s == 1) ? 0.0 : val_bit
        # El cierre colapsa el enlace virtual multiplicando por el vector de salida [acumulado, 1.0]^T
        # Esto extrae el acumulado final del canal 2 y le suma la última contribución local
        b[N][blinks[N-1]=>1, sites[N]=>s] = contribucion #Fila 1
        b[N][blinks[N-1]=>2, sites[N]=>s] = 1.0 #Fila 2
    end
    
    return b
end


construir_lineal_manual (generic function with 2 methods)

In [61]:
# --- PARÁMETROS DEL SISTEMA ---
N = 10
dim_total = 1 << N #dim_total=2^N
x_min = 1.0
x_max = 2.0  # Usamos [0, 1]
dx = (x_max - x_min) / dim_total

#Sitios necesarios para obtener QTT
sites = siteinds("Qubit", N)

# Construimos el mps a partir del vector denso
mps_svd=construir_lineal_SVD(sites,x_min,dx,N)

# Construimos el mps hecho a mano
mps_manual = construir_lineal_manual(sites, x_min,dx, N)

# ==========================================================
# EVALUACIÓN EN EL BANCO DE PRUEBAS
# ==========================================================
distancia=evaluar_mps_manual(N, mps_svd, mps_manual)

if distancia < 1e-6
  println("🎉 ¡ÉXITO! Tu construcción manual es exacta.")
else
  println("❌ Hay un error en la lógica de los tensores manuales.")
end
println("Max Bond Dim (SVD):    ", maxlinkdim(mps_svd))
println("Max Bond Dim (Manual): ", maxlinkdim(mps_manual))
println("Distancia (Error):     ", distancia)

🎉 ¡ÉXITO! Tu construcción manual es exacta.
Max Bond Dim (SVD):    2
Max Bond Dim (Manual): 2
Distancia (Error):     9.5367431640625e-7


Siguiente reto: construcción de la función f(x)=x^2, también de dos maneras.

In [62]:
# ==========================================================
# Construcción mediante vector denso con SVD, para f(x)=x^2
# ==========================================================
function construir_cuadratica_SVD(sites,x_min,dx,N)
  psi_tensor = ITensor(sites...)
  for i in 0:(dim_total - 1)
    x = x_min + i * dx
    val = x^2  # f(x) = x^2
    estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
    psi_tensor[estado_binario...] = val
  end
  mps_svd = MPS(psi_tensor, sites; cutoff=1e-15)
  return mps_svd
end

construir_cuadratica_SVD (generic function with 2 methods)

In [63]:
# ==========================================================
# CONSTRUCCIÓN MANUAL DE QTT PARA f(x) = x² (χ = 3)
# ==========================================================
function construir_cuadratica_manual(sites, x_min,dx, N)
    b = MPS(sites)
    
    # Definimos los enlaces virtuales con dimensión fija χ = 3
    blinks = [Index(3, "link,l=$i") for i in 1:(N-1)]
    
    # --- SITIO 1: Vector Fila (3 valores) ---
    b[1] = ITensor(sites[1], blinks[1])
    for s in 1:2
        c = (s == 1) ? 0.0 : dx * 2.0^0
        b[1][sites[1]=>s, blinks[1]=>1] = 1.0
        b[1][sites[1]=>s, blinks[1]=>2] = x_min+c
        b[1][sites[1]=>s, blinks[1]=>3] = (x_min+c)^2
    end

    # --- BULK: Matriz 3x3 (Sitios 2 a N-1) ---
    for k in 2:(N-1)
        b[k] = ITensor(blinks[k-1], sites[k], blinks[k])
        val_bit = dx * 2.0^(k-1)
        for s in 1:2
            c = (s == 1) ? 0.0 : val_bit
            
            # Fila 1: Entrada desde canal neutro (1)
            b[k][blinks[k-1]=>1, sites[k]=>s, blinks[k]=>1] = 1.0
            b[k][blinks[k-1]=>1, sites[k]=>s, blinks[k]=>2] = c
            b[k][blinks[k-1]=>1, sites[k]=>s, blinks[k]=>3] = c^2
            
            # Fila 2: Entrada desde canal lineal (X)
            b[k][blinks[k-1]=>2, sites[k]=>s, blinks[k]=>1] = 0.0
            b[k][blinks[k-1]=>2, sites[k]=>s, blinks[k]=>2] = 1.0
            b[k][blinks[k-1]=>2, sites[k]=>s, blinks[k]=>3] = 2 * c
            
            # Fila 3: Entrada desde canal cuadrático (X²)
            b[k][blinks[k-1]=>3, sites[k]=>s, blinks[k]=>1] = 0.0
            b[k][blinks[k-1]=>3, sites[k]=>s, blinks[k]=>2] = 0.0
            b[k][blinks[k-1]=>3, sites[k]=>s, blinks[k]=>3] = 1.0
        end
    end

    # --- SITIO N: Vector Columna de Cierre (3 valores) ---
    b[N] = ITensor(blinks[N-1], sites[N])
    val_bit = dx * 2.0^(N-1)
    for s in 1:2
        c = (s == 1) ? 0.0 : val_bit
        
        # Cierre colapsando la multiplicación en la tercera componente (X²)
        b[N][blinks[N-1]=>1, sites[N]=>s] = c^2
        b[N][blinks[N-1]=>2, sites[N]=>s] = 2 * c
        b[N][blinks[N-1]=>3, sites[N]=>s] = 1.0
    end
    
    return b
end


construir_cuadratica_manual (generic function with 2 methods)

In [65]:
# --- PARÁMETROS DEL SISTEMA ---
N = 10
dim_total = 1 << N #dim_total=2^N
x_min = 1.0
x_max = 2.0  # Usamos [0, 1]
dx = (x_max - x_min) / dim_total

#Sitios necesarios para obtener QTT
sites = siteinds("Qubit", N)

# Construimos el mps a partir del vector denso
mps_svd=construir_cuadratica_SVD(sites,x_min,dx,N)

# Construimos el mps hecho a mano
mps_manual = construir_cuadratica_manual(sites, x_min,dx, N)

# ==========================================================
# EVALUACIÓN EN EL BANCO DE PRUEBAS
# ==========================================================
distancia=evaluar_mps_manual(N, mps_svd, mps_manual)

if distancia < 1e-5
  println("🎉 ¡ÉXITO! Tu construcción manual es exacta.")
else
  println("❌ Hay un error en la lógica de los tensores manuales.")
end
println("Max Bond Dim (SVD):    ", maxlinkdim(mps_svd))
println("Max Bond Dim (Manual): ", maxlinkdim(mps_manual))
println("Distancia (Error):     ", distancia)

🎉 ¡ÉXITO! Tu construcción manual es exacta.
Max Bond Dim (SVD):    3
Max Bond Dim (Manual): 3
Distancia (Error):     2.6973983046972182e-6


El siguiente paso es una función exponencial. Como antes, mediante los dos métodos

In [66]:
# ==========================================================
# Construcción mediante vector denso con SVD, para f(x)=e^(alpha x)
# ==========================================================
function construir_exponencial_SVD(sites,x_min,dx,N,alpha)
  psi_tensor = ITensor(sites...)
  for i in 0:(dim_total - 1)
    x = x_min + i * dx
    val = exp(alpha * x)  # f(x) = e^(alpha x)
    estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
    psi_tensor[estado_binario...] = val
  end
  mps_svd = MPS(psi_tensor, sites; cutoff=1e-15)
  return mps_svd
end

construir_exponencial_SVD (generic function with 2 methods)

In [67]:
# ==========================================================
# Construcción analítica a mano (Rango χ = 1)
# ==========================================================
function construir_exponencial_manual(sites, x_min,dx, N, alpha)
    b = MPS(sites)
    
    # No hay 'blinks' (enlaces virtuales). 
    # Cada nodo es un tensor de rango 1 que solo posee el índice físico del qubit.
    
    for k in 1:N
        b[k] = ITensor(sites[k])
        val_bit = dx * 2.0^(k-1)
        
        # Estado |0> (s = 1): c = 0.0  => e^(α * 0) = 1.0
        b[k][sites[k]=>1] = 1.0
        
        # Estado |1> (s = 2): c = val_bit => e^(α * c)
        b[k][sites[k]=>2] = exp(alpha * val_bit)
    end

    # Al final, aplicamos el offset globalmente
    b[1] *= exp(alpha * x_min)# Multiplicar un solo sitio escala todo el MPS
    return b
end

construir_exponencial_manual (generic function with 2 methods)

In [70]:
# --- PARÁMETROS DEL SISTEMA ---
N = 10
dim_total = 1 << N #dim_total=2^N
x_min = 1.0
x_max = 2.0  # Usamos [0, 1]
dx = (x_max - x_min) / dim_total

#Sitios necesarios para obtener QTT
sites = siteinds("Qubit", N)

# Parámetro de la exponencial: f(x) = e^(α * x)
alpha = 2.5

# Construimos el mps a partir del vector denso
mps_svd=construir_exponencial_SVD(sites,x_min,dx,N,alpha)

# Construimos el mps hecho a mano
mps_manual = construir_exponencial_manual(sites, x_min,dx, N,alpha)

# ==========================================================
# EVALUACIÓN EN EL BANCO DE PRUEBAS
# ==========================================================
distancia=evaluar_mps_manual(N, mps_svd, mps_manual)

if distancia < 1e-10
  println("🎉 ¡ÉXITO! Tu construcción manual es exacta.")
else
  println("❌ Hay un error en la lógica de los tensores manuales.")
end
println("Max Bond Dim (SVD):    ", maxlinkdim(mps_svd))
println("Max Bond Dim (Manual): ", maxlinkdim(mps_manual))
println("Distancia (Error):     ", distancia)

🎉 ¡ÉXITO! Tu construcción manual es exacta.
Max Bond Dim (SVD):    1
Max Bond Dim (Manual): 1
Distancia (Error):     0.0


El siguiente reto son las funciones trigonométricas. Hay dos enfoques: exponenciales complejas y matrices de rotación. En primer lugar se va a hacer con exponeciales complejas, para la funcion f(x)=cos(n x). Como siempre, usando los dos métods, SVD y manual.

In [71]:
# ==========================================================
# Construcción mediante vector denso con SVD, para f(x)=cos(n x)
# ==========================================================
function construir_coseno_SVD(sites,x_min,dx,N,n)
  psi_tensor = ITensor(sites...)
  for i in 0:(dim_total - 1)
    x = x_min + i * dx
    val = cos(n * x)  # f(x) = cos(n x)
    estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
    psi_tensor[estado_binario...] = val
  end
  mps_svd = MPS(psi_tensor, sites; cutoff=1e-15)
  return mps_svd
end

construir_coseno_SVD (generic function with 2 methods)

In [72]:
# ==========================================================
# Construcción analítica a mano (Rango χ = 2) usando exponenciales complejas
# ==========================================================
function construir_coseno_manual(sites, x_min,dx, N, n)
    b = MPS(sites)
    
    # Enlaces virtuales de dimensión fija χ = 2
    blinks = [Index(2, "link,l=$i") for i in 1:(N-1)]
    
    # --- SITIO 1: Vector Fila (Inyecta el factor 1/2 de Euler) ---
    b[1] = ITensor(ComplexF64, sites[1], blinks[1])
    for s in 1:2
        c = (s == 1) ? 0.0 : n * dx * 2.0^0
        # Canal 1: Inyecta 0.5 * e^(i * n * x_min) * e^(i * c)
        b[1][sites[1]=>s, blinks[1]=>1] = 0.5 * exp(im * n * x_min) * exp(im * c)
        # Canal 2: Inyecta 0.5 * e^(-i * n * x_min) * e^(-i * c)
        b[1][sites[1]=>s, blinks[1]=>2] = 0.5 * exp(-im * n * x_min) * exp(-im * c)
    end

    # --- BULK: Matrices Diagonales 2x2 (Sitios 2 a N-1) ---
    for k in 2:(N-1)
        b[k] = ITensor(ComplexF64, blinks[k-1], sites[k], blinks[k])
        val_bit = n * dx * 2.0^(k-1)
        for s in 1:2
            c = (s == 1) ? 0.0 : val_bit
            
            # Fila 1: Propaga la fase positiva e^(i*c) en el Canal 1
            b[k][blinks[k-1]=>1, sites[k]=>s, blinks[k]=>1] = exp(im * c)
            b[k][blinks[k-1]=>1, sites[k]=>s, blinks[k]=>2] = 0.0
            
            # Fila 2: Propaga la fase negativa e^(-i*c) en el Canal 2
            b[k][blinks[k-1]=>2, sites[k]=>s, blinks[k]=>1] = 0.0
            b[k][blinks[k-1]=>2, sites[k]=>s, blinks[k]=>2] = exp(-im * c)
        end
    end

    # --- SITIO N: Vector Columna (Cierre neutro que suma ambos canales) ---
    b[N] = ITensor(ComplexF64, blinks[N-1], sites[N])
    val_bit = n * dx * 2.0^(N-1)
    for s in 1:2
        c = (s == 1) ? 0.0 : val_bit
        
        # Al colapsar contra [1, 1]ᵀ multiplicando la última matriz del bulk:
        # Fila 1 recibe su última fase e^(i*c)
        b[N][blinks[N-1]=>1, sites[N]=>s] = exp(im * c)
        # Fila 2 recibe su última fase e^(-i*c)
        b[N][blinks[N-1]=>2, sites[N]=>s] = exp(-im * c)
    end
    
    return b
end

construir_coseno_manual (generic function with 2 methods)

In [74]:
# --- PARÁMETROS DEL SISTEMA ---
N = 10
dim_total = 1 << N #dim_total=2^N
x_min = 1.0
x_max = 2.0  # Usamos [0, 1]
dx = (x_max - x_min) / dim_total

#Sitios necesarios para obtener QTT
sites = siteinds("Qubit", N)

# Parámetro del coseno: f(x) = cos(n x)
n_freq = 15.5

# Construimos el mps a partir del vector denso
mps_svd=construir_coseno_SVD(sites,x_min,dx,N,n_freq)

# Construimos el mps hecho a mano
mps_manual = construir_coseno_manual(sites,x_min, dx, N,n_freq)

# ==========================================================
# EVALUACIÓN EN EL BANCO DE PRUEBAS
# ==========================================================
distancia=evaluar_mps_manual(N, mps_svd, mps_manual)

if distancia < 1e-10
  println("🎉 ¡ÉXITO! Tu construcción manual es exacta.")
else
  println("❌ Hay un error en la lógica de los tensores manuales.")
end
println("Max Bond Dim (SVD):    ", maxlinkdim(mps_svd))
println("Max Bond Dim (Manual): ", maxlinkdim(mps_manual))
println("Distancia (Error):     ", distancia)

🎉 ¡ÉXITO! Tu construcción manual es exacta.
Max Bond Dim (SVD):    2
Max Bond Dim (Manual): 2
Distancia (Error):     0.0


El siguiente reto es la construcción del coseno mediante matrices de rotación, bit a bit, basadas en la propiedades del seno y coseno de la suma.

In [75]:
# ==========================================================
# Coseno Analítico mediante Matrices de Rotación (χ = 2)
# ==========================================================
function construir_coseno_rotacion(sites, x_min,dx, N, k_freq)
    b = MPS(sites)
    blinks = [Index(2, "link,l=$i") for i in 1:(N-1)]
    
    # --- SITIO 1: Inyección Inicial con x_min ---
    b[1] = ITensor(sites[1], blinks[1])
    th_min = k_freq * x_min
    th1 = k_freq * dx * 2.0^0

    # s=1 (bit=0 -> No hay rotación extra, se queda en th_min)
    b[1][sites[1]=>1, blinks[1]=>1] = cos(th_min)
    b[1][sites[1]=>1, blinks[1]=>2] = sin(th_min)

    # s=2 (bit=1 -> Rotamos th_min un ángulo th1 extra)
    # Usamos las identidades: cos(A+B) = cosA*cosB - sinA*sinB
    #                        sin(A+B) = sinA*cosB + cosA*sinB
    b[1][sites[1]=>2, blinks[1]=>1] = cos(th_min + th1)
    b[1][sites[1]=>2, blinks[1]=>2] = sin(th_min + th1)

    # --- BULK: Sitios 2 a N-1 (Matrices de Rotación) ---
    for k in 2:(N-1)
        b[k] = ITensor(blinks[k-1], sites[k], blinks[k])
        th = k_freq * dx * 2.0^(k-1)
        
        # s=1 (bit=0): Matriz Identidad Virtual
        b[k][blinks[k-1]=>1, sites[k]=>1, blinks[k]=>1] = 1.0
        b[k][blinks[k-1]=>2, sites[k]=>1, blinks[k]=>2] = 1.0
        
        # s=2 (bit=1): Matriz de Rotación Horaria/Antihoraria
        b[k][blinks[k-1]=>1, sites[k]=>2, blinks[k]=>1] = cos(th)
        b[k][blinks[k-1]=>1, sites[k]=>2, blinks[k]=>2] = sin(th)
        b[k][blinks[k-1]=>2, sites[k]=>2, blinks[k]=>1] = -sin(th)
        b[k][blinks[k-1]=>2, sites[k]=>2, blinks[k]=>2] = cos(th)
    end

    # --- SITIO N: Cierre y Extracción del Coseno ---
    b[N] = ITensor(blinks[N-1], sites[N])
    thN = k_freq * dx * 2.0^(N-1)
    
    # s=1 (bit=0): Multiplica por vector c = [1, 0]^T
    b[N][blinks[N-1]=>1, sites[N]=>1] = 1.0
    b[N][blinks[N-1]=>2, sites[N]=>1] = 0.0
    
    # s=2 (bit=1): Rota y luego extrae la primera componente
    b[N][blinks[N-1]=>1, sites[N]=>2] = cos(thN)
    b[N][blinks[N-1]=>2, sites[N]=>2] = -sin(thN)
    
    return b
end

construir_coseno_rotacion (generic function with 2 methods)

In [77]:
# --- PARÁMETROS DEL SISTEMA ---
N = 10
dim_total = 1 << N #dim_total=2^N
x_min = 1.0
x_max = 2.0  # Usamos [0, 1]
dx = (x_max - x_min) / dim_total

#Sitios necesarios para obtener QTT
sites = siteinds("Qubit", N)

# Parámetro del coseno: f(x) = cos(n x)
n_freq = 15.5

# Construimos el mps a partir del vector denso
mps_svd=construir_coseno_SVD(sites,x_min,dx,N,n_freq)

# Construimos el mps hecho a mano
mps_manual = construir_coseno_rotacion(sites, x_min,dx, N,n_freq)

# ==========================================================
# EVALUACIÓN EN EL BANCO DE PRUEBAS
# ==========================================================
distancia=evaluar_mps_manual(N, mps_svd, mps_manual)

if distancia < 1e-10
  println("🎉 ¡ÉXITO! Tu construcción manual es exacta.")
else
  println("❌ Hay un error en la lógica de los tensores manuales.")
end
println("Max Bond Dim (SVD):    ", maxlinkdim(mps_svd))
println("Max Bond Dim (Manual): ", maxlinkdim(mps_manual))
println("Distancia (Error):     ", distancia)

🎉 ¡ÉXITO! Tu construcción manual es exacta.
Max Bond Dim (SVD):    2
Max Bond Dim (Manual): 2
Distancia (Error):     0.0


La misma idea pero con el seno, también mediante rotación

In [81]:
# ==========================================================
# Construcción mediante vector denso con SVD, para f(x)=sin(n x)
# ==========================================================
function construir_seno_SVD(sites,x_min,dx,N,n)
  psi_tensor = ITensor(sites...)
  for i in 0:(dim_total - 1)
    x = x_min + i * dx
    val = sin(n * x)  # f(x) = sin(n x)
    estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
    psi_tensor[estado_binario...] = val
  end
  mps_svd = MPS(psi_tensor, sites; cutoff=1e-15)
  return mps_svd
end

construir_seno_SVD (generic function with 2 methods)

In [82]:
# ==========================================================
# Seno Analítico mediante Matrices de Rotación (χ = 2)
# ==========================================================
function construir_seno_rotacion(sites, x_min,dx, N, k_freq)
    b = MPS(sites)
    blinks = [Index(2, "link,l=$i") for i in 1:(N-1)]
    
    # --- SITIO 1: Inyección Inicial con x_min ---
    b[1] = ITensor(sites[1], blinks[1])
    th_min = k_freq * x_min
    th1 = k_freq * dx * 2.0^0

    # s=1 (bit=0 -> No hay rotación extra, se queda en th_min)
    b[1][sites[1]=>1, blinks[1]=>1] = cos(th_min)
    b[1][sites[1]=>1, blinks[1]=>2] = sin(th_min)

    # s=2 (bit=1 -> Rotamos th_min un ángulo th1 extra)
    # Usamos las identidades: cos(A+B) = cosA*cosB - sinA*sinB
    #                        sin(A+B) = sinA*cosB + cosA*sinB
    b[1][sites[1]=>2, blinks[1]=>1] = cos(th_min + th1)
    b[1][sites[1]=>2, blinks[1]=>2] = sin(th_min + th1)


    # --- BULK: Sitios 2 a N-1 (Matrices de Rotación) ---
    for k in 2:(N-1)
        b[k] = ITensor(blinks[k-1], sites[k], blinks[k])
        th = k_freq * dx * 2.0^(k-1)
        
        # s=1 (bit=0): Matriz Identidad Virtual (No añade ángulo)
        b[k][blinks[k-1]=>1, sites[k]=>1, blinks[k]=>1] = 1.0
        b[k][blinks[k-1]=>2, sites[k]=>1, blinks[k]=>2] = 1.0
        
        # s=2 (bit=1): Matriz de Rotación pura
        b[k][blinks[k-1]=>1, sites[k]=>2, blinks[k]=>1] = cos(th)
        b[k][blinks[k-1]=>1, sites[k]=>2, blinks[k]=>2] = sin(th)
        b[k][blinks[k-1]=>2, sites[k]=>2, blinks[k]=>1] = -sin(th)
        b[k][blinks[k-1]=>2, sites[k]=>2, blinks[k]=>2] = cos(th)
    end

    # --- SITIO N: Cierre y Extracción del SENO ---
    b[N] = ITensor(blinks[N-1], sites[N])
    thN = k_freq * dx * 2.0^(N-1)
    
    # s=1 (bit=0): Multiplica por vector c = [0, 1]^T para extraer el CANAL 2
    b[N][blinks[N-1]=>1, sites[N]=>1] = 0.0
    b[N][blinks[N-1]=>2, sites[N]=>1] = 1.0
    
    # s=2 (bit=1): Rota el espacio virtual y extrae la segunda componente
    # Usando el vector c = [0, 1]^T tras la multiplicación:
    b[N][blinks[N-1]=>1, sites[N]=>2] = sin(thN)
    b[N][blinks[N-1]=>2, sites[N]=>2] = cos(thN)
    
    return b
end

construir_seno_rotacion (generic function with 2 methods)

In [83]:
# --- PARÁMETROS DEL SISTEMA ---
N = 10
dim_total = 1 << N #dim_total=2^N
x_min = 1.0
x_max = 2.0  # Usamos [0, 1]
dx = (x_max - x_min) / dim_total

#Sitios necesarios para obtener QTT
sites = siteinds("Qubit", N)

# Parámetro del seno: f(x) = sin(n x)
n_freq = 15.5

# Construimos el mps a partir del vector denso
mps_svd=construir_seno_SVD(sites,x_min,dx,N,n_freq)

# Construimos el mps hecho a mano
mps_manual = construir_seno_rotacion(sites, x_min,dx, N,n_freq)

# ==========================================================
# EVALUACIÓN EN EL BANCO DE PRUEBAS
# ==========================================================
distancia=evaluar_mps_manual(N, mps_svd, mps_manual)

if distancia < 1e-10
  println("🎉 ¡ÉXITO! Tu construcción manual es exacta.")
else
  println("❌ Hay un error en la lógica de los tensores manuales.")
end
println("Max Bond Dim (SVD):    ", maxlinkdim(mps_svd))
println("Max Bond Dim (Manual): ", maxlinkdim(mps_manual))
println("Distancia (Error):     ", distancia)

🎉 ¡ÉXITO! Tu construcción manual es exacta.
Max Bond Dim (SVD):    2
Max Bond Dim (Manual): 2
Distancia (Error):     0.0


El siguiente reto es la generación de funciones hiperbólicas mediante matrices de "rotación", basadas en la propiedades del seno y coseno de la suma.

In [84]:
# ==========================================================
# Construcción mediante vector denso con SVD, para f(x)=cosh(n x)
# ==========================================================
function construir_cosh_SVD(sites,x_min,dx,N,n)
  psi_tensor = ITensor(sites...)
  for i in 0:(dim_total - 1)
    x = x_min + i * dx
    val = cosh(n * x)  # f(x) = cosh(n x)
    estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
    psi_tensor[estado_binario...] = val
  end
  mps_svd = MPS(psi_tensor, sites; cutoff=1e-15)
  return mps_svd
end

construir_cosh_SVD (generic function with 1 method)

In [85]:
# ==========================================================
# Funciones Hiperbólicas por Rotación de Lorentz (χ = 2)
# ==========================================================
function construir_hiperbolica_rotacion(sites, x_min,dx, N, k_scale; tipo="cosh")
    b = MPS(sites)
    blinks = [Index(2, "link,l=$i") for i in 1:(N-1)]

    # --- SITIO 1: Inyección Inicial con x_min ---
    b[1] = ITensor(sites[1], blinks[1])
    th_min = k_scale * x_min
    th1 = k_scale * dx * 2.0^0

    # s=1 (bit=0 -> No hay rotación extra, se queda en th_min)
    b[1][sites[1]=>1, blinks[1]=>1] = cosh(th_min)
    b[1][sites[1]=>1, blinks[1]=>2] = sinh(th_min)

    # s=2 (bit=1 -> Rotamos th_min un ángulo th1 extra)
    # Usamos las identidades: cosh(A+B) = coshA*coshB + sinhA*sinhB
    #                        sinh(A+B) = sinhA*coshB + coshA*sinhB
    b[1][sites[1]=>2, blinks[1]=>1] = cosh(th_min + th1)
    b[1][sites[1]=>2, blinks[1]=>2] = sinh(th_min + th1)

    # --- BULK: Sitios 2 a N-1 (Matrices de Boost de Lorentz) ---
    for k in 2:(N-1)
        b[k] = ITensor(blinks[k-1], sites[k], blinks[k])
        th = k_scale * dx * 2.0^(k-1)
        
        # s=1 (bit=0): Identidad (No añade argumento)
        b[k][blinks[k-1]=>1, sites[k]=>1, blinks[k]=>1] = 1.0
        b[k][blinks[k-1]=>2, sites[k]=>1, blinks[k]=>2] = 1.0
        
        # s=2 (bit=1): Rotación Hiperbólica (Matriz simétrica)
        b[k][blinks[k-1]=>1, sites[k]=>2, blinks[k]=>1] = cosh(th)
        b[k][blinks[k-1]=>1, sites[k]=>2, blinks[k]=>2] = sinh(th)
        b[k][blinks[k-1]=>2, sites[k]=>2, blinks[k]=>1] = sinh(th)
        b[k][blinks[k-1]=>2, sites[k]=>2, blinks[k]=>2] = cosh(th)
    end

    # --- SITIO N: Cierre y Selección de Canal ---
    b[N] = ITensor(blinks[N-1], sites[N])
    thN = k_scale * dx * 2.0^(N-1)
    
    if tipo == "cosh"
        # Extrae Canal 1 (Multiplica por c = [1, 0]^T)
        b[N][blinks[N-1]=>1, sites[N]=>1] = 1.0
        b[N][blinks[N-1]=>2, sites[N]=>1] = 0.0
        
        b[N][blinks[N-1]=>1, sites[N]=>2] = cosh(thN)
        b[N][blinks[N-1]=>2, sites[N]=>2] = sinh(thN)
    elseif tipo == "sinh"
        # Extrae Canal 2 (Multiplica por c = [0, 1]^T)
        b[N][blinks[N-1]=>1, sites[N]=>1] = 0.0
        b[N][blinks[N-1]=>2, sites[N]=>1] = 1.0
        
        b[N][blinks[N-1]=>1, sites[N]=>2] = sinh(thN)
        b[N][blinks[N-1]=>2, sites[N]=>2] = cosh(thN)
    end
    
    return b
end

construir_hiperbolica_rotacion (generic function with 1 method)

In [88]:
# --- PARÁMETROS DEL SISTEMA ---
N = 10
dim_total = 1 << N #dim_total=2^N
x_min = 1.0
x_max = 2.0  # Usamos [0, 1]
dx = (x_max - x_min) / dim_total

#Sitios necesarios para obtener QTT
sites = siteinds("Qubit", N)

# Parámetro del seno: f(x) = sin(n x)
k_scale = 1.5

# Construimos el mps a partir del vector denso
mps_svd=construir_cosh_SVD(sites,x_min,dx,N,k_scale)

# Construimos el mps hecho a mano
mps_manual = construir_hiperbolica_rotacion(sites, x_min,dx, N, k_scale; tipo="cosh")

# ==========================================================
# EVALUACIÓN EN EL BANCO DE PRUEBAS
# ==========================================================
distancia=evaluar_mps_manual(N, mps_svd, mps_manual)

if distancia < 1e-10
  println("🎉 ¡ÉXITO! Tu construcción manual es exacta.")
else
  println("❌ Hay un error en la lógica de los tensores manuales.")
end
println("Max Bond Dim (SVD):    ", maxlinkdim(mps_svd))
println("Max Bond Dim (Manual): ", maxlinkdim(mps_manual))
println("Distancia (Error):     ", distancia)

🎉 ¡ÉXITO! Tu construcción manual es exacta.
Max Bond Dim (SVD):    2
Max Bond Dim (Manual): 2
Distancia (Error):     0.0
